# Дополнительные задания

Добавить в конец основного ноутбука после загрузки модели.

Предполагается, что уже выполнены ячейки:
- Генератор, BFS, VecMazeEnv, MazeCNN, PPO
- Данные (train_mazes, eval_mazes, train_dist, eval_dist, train_sp, eval_sp)
- Модель загружена

---
## 9.1 Параллельная среда с авто-рестартом (до +2)

`VecMazeEnv` поддерживает:
- Одновременное выполнение N_ENVS игр параллельно
- Автоматический рестарт завершённых игр (`auto_reset_done()`)
- Все операции в `step()` и `reset()` — numpy-only

Это используется при обучении (N_ENVS=64). Ниже — наглядная демонстрация.

In [ ]:
# === Демонстрация параллельной среды с авто-рестартом ===

n_demo = 8
demo_env = VecMazeEnv(eval_mazes, n_envs=n_demo, max_steps=400,
                      stage=1, dist_pool=eval_dist, sp_pool=eval_sp)
obs = demo_env.reset()
print(f'Параллельных сред: {n_demo}')
print(f'Obs shape: {obs.shape}\n')

total_episodes = 0
total_exits = 0

for step in range(500):
    with torch.no_grad():
        actions, _, _, _ = model.act(torch.tensor(obs, device=device))
        actions = actions.cpu().numpy()

    obs, rewards, dones, infos = demo_env.step(actions)

    for i in range(n_demo):
        if infos['reached_exit'][i] or infos['timeout'][i]:
            total_episodes += 1
            if infos['reached_exit'][i]:
                total_exits += 1
            if total_episodes <= 8:
                status = 'EXIT' if infos['reached_exit'][i] else 'TIMEOUT'
                print(f'  Step {step:3d}: Env #{i} → {status} '
                      f'({infos["steps"][i]} steps) → auto-reset')

    obs = demo_env.auto_reset_done()

print(f'\n--- Итого за 500 шагов в {n_demo} средах ---')
print(f'Завершено эпизодов: {total_episodes}')
print(f'Дошли до выхода: {total_exits} ({total_exits/max(total_episodes,1):.0%})')

---
## 9.2 Визуализация промежуточных выходов CNN (до +1)

Видео прохождения лабиринта с отображением:
- Текущее состояние лабиринта с путём агента
- Feature maps свёрточных слоёв (что «видит» модель)
- Вероятности действий (policy output)

In [ ]:
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 50  # MB
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def make_maze_video(model, maze, sp_mask=None, max_steps=200, fps=5):
    """
    Создаёт видео прохождения лабиринта с визуализацией
    промежуточных выходов CNN (feature maps).
    """
    mf = maze.astype(np.float32)

    # --- Собираем траекторию ---
    r, c = 1, 0
    frames_data = []  # (r, c, path, visited, activations, probs, value)
    visited = np.zeros((21, 21), dtype=np.float32)
    visited[r, c] = 1.0
    path = [(r, c)]

    for step in range(max_steps):
        obs = np.zeros((1, 6, 21, 21), dtype=np.float32)
        obs[0,0]=mf; obs[0,1,r,c]=1.0; obs[0,2,19,20]=1.0
        obs[0,3]=visited; obs[0,4]=COORD_R; obs[0,5]=COORD_C
        obs_t = torch.tensor(obs, device=device)

        # Извлечь feature maps
        acts = []
        x = obs_t
        with torch.no_grad():
            for layer in model.conv:
                x = layer(x)
                if isinstance(layer, nn.ReLU):
                    acts.append(x.cpu().numpy()[0])

            logits, value = model(obs_t)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
            val = value.item()
            action = Categorical(logits=logits).sample().item()

        frames_data.append({
            'r': r, 'c': c,
            'path': list(path),
            'visited': visited.copy(),
            'acts': acts,
            'probs': probs,
            'value': val,
            'action': action,
        })

        if r == 19 and c == 20:
            break

        dr = [1,0,-1,0][action]; dc = [0,1,0,-1][action]
        nr, nc = r+dr, c+dc
        if 0<=nr<21 and 0<=nc<21 and maze[nr,nc]==1:
            r, c = nr, nc
        visited[r,c] = 1.0
        path.append((r,c))

    print(f'Trajectory: {len(frames_data)} steps, '
          f'{"reached exit" if (r==19 and c==20) else "did not reach"}')

    # --- Создаём видео ---
    # Layout: top=maze, middle=feature maps (2x2), bottom=policy
    fig = plt.figure(figsize=(12, 10))

    # Axes
    ax_maze = fig.add_axes([0.05, 0.55, 0.4, 0.4])   # maze
    ax_fm = []  # 2x2 feature maps
    for row in range(2):
        for col in range(2):
            ax = fig.add_axes([0.5 + col*0.25, 0.55 + (1-row)*0.2, 0.22, 0.18])
            ax_fm.append(ax)
    ax_pol = fig.add_axes([0.05, 0.05, 0.25, 0.4])   # policy
    ax_fm_big = []  # 3 bottom feature maps
    for col in range(3):
        ax = fig.add_axes([0.35 + col*0.22, 0.05, 0.2, 0.4])
        ax_fm_big.append(ax)

    action_names = ['Down', 'Right', 'Up', 'Left']
    action_arrows = ['↓', '→', '↑', '←']
    layer_names = ['Conv1 32ch\n21×21', 'Conv2 64ch\n11×11',
                   'Conv3 64ch\n6×6', 'Conv4 128ch\n3×3']

    def update(frame_idx):
        fd = frames_data[frame_idx]

        # --- Maze ---
        ax_maze.clear()
        img = np.stack([maze.astype(float)]*3, axis=-1)
        # Draw path
        for pr, pc in fd['path']:
            img[pr, pc] = [0.7, 0.7, 0.7]
        # Current position
        img[fd['r'], fd['c']] = [0, 0, 1]  # blue
        img[19, 20] = [0, 1, 0]  # green exit
        img[1, 0] = [1, 1, 0]    # yellow entry
        ax_maze.imshow(img)
        ax_maze.set_title(f'Step {frame_idx} | Pos ({fd["r"]},{fd["c"]}) | '
                         f'{action_arrows[fd["action"]]}', fontsize=10)
        ax_maze.set_xticks([]); ax_maze.set_yticks([])

        # --- Feature maps (top 2x2: conv layers 1-4, most active filter) ---
        for li in range(min(4, len(fd['acts']))):
            ax_fm[li].clear()
            act = fd['acts'][li]
            # Most active filter
            best = act.reshape(act.shape[0], -1).max(axis=1).argmax()
            ax_fm[li].imshow(act[best], cmap='viridis', interpolation='nearest')
            ax_fm[li].set_title(layer_names[li], fontsize=7)
            ax_fm[li].set_xticks([]); ax_fm[li].set_yticks([])

        # --- Bottom: 3 more filters from conv1 ---
        act0 = fd['acts'][0]  # conv1, 32 channels, 21x21
        activity = act0.reshape(act0.shape[0], -1).max(axis=1)
        top3 = np.argsort(activity)[-4:-1][::-1]  # 2nd, 3rd, 4th most active
        for fi, filt_idx in enumerate(top3):
            ax_fm_big[fi].clear()
            ax_fm_big[fi].imshow(act0[filt_idx], cmap='viridis', interpolation='nearest')
            ax_fm_big[fi].set_title(f'Conv1 filter {filt_idx}', fontsize=8)
            ax_fm_big[fi].set_xticks([]); ax_fm_big[fi].set_yticks([])

        # --- Policy ---
        ax_pol.clear()
        colors = ['#e74c3c' if i==fd['action'] else '#3498db' for i in range(4)]
        ax_pol.barh(action_names, fd['probs'], color=colors)
        ax_pol.set_xlim(0, 1)
        ax_pol.set_title(f'Policy | V={fd["value"]:.1f}', fontsize=9)
        for i, p in enumerate(fd['probs']):
            ax_pol.text(p + 0.02, i, f'{p:.0%}', va='center', fontsize=8)

    anim = FuncAnimation(fig, update, frames=len(frames_data),
                         interval=1000//fps, repeat=True)
    plt.close(fig)
    return anim

print('Video function ready')

In [ ]:
# Генерация видео для 2 лабиринтов
for _ in range(2):
    idx = np.random.randint(len(eval_mazes))
    print(f'\nMaze #{idx}:')
    anim = make_maze_video(model, eval_mazes[idx], eval_sp[idx], max_steps=150, fps=4)
    display(HTML(anim.to_jshtml()))

In [ ]:
# Статичная визуализация feature maps (один кадр)
def visualize_cnn_features_static(model, maze, step_num=None):
    mf = maze.astype(np.float32)
    r, c = 1, 0
    visited = np.zeros((21,21), dtype=np.float32); visited[r,c]=1.0
    path = [(r,c)]

    n_warmup = step_num if step_num else random.randint(10, 40)
    for _ in range(n_warmup):
        if r==19 and c==20: break
        obs=np.zeros((1,6,21,21),dtype=np.float32)
        obs[0,0]=mf;obs[0,1,r,c]=1.0;obs[0,2,19,20]=1.0
        obs[0,3]=visited;obs[0,4]=COORD_R;obs[0,5]=COORD_C
        with torch.no_grad():
            act,_,_,_=model.act(torch.tensor(obs,device=device))
            action=act.item()
        dr=[1,0,-1,0][action];dc=[0,1,0,-1][action]
        nr,nc=r+dr,c+dc
        if 0<=nr<21 and 0<=nc<21 and maze[nr,nc]==1: r,c=nr,nc
        visited[r,c]=1.0; path.append((r,c))

    obs=np.zeros((1,6,21,21),dtype=np.float32)
    obs[0,0]=mf;obs[0,1,r,c]=1.0;obs[0,2,19,20]=1.0
    obs[0,3]=visited;obs[0,4]=COORD_R;obs[0,5]=COORD_C
    obs_t=torch.tensor(obs,device=device)

    acts=[]; x=obs_t
    with torch.no_grad():
        for layer in model.conv:
            x=layer(x)
            if isinstance(layer,nn.ReLU): acts.append(x.cpu().numpy()[0])
        logits,value=model(obs_t)
        probs=torch.softmax(logits,-1).cpu().numpy()[0]

    fig,axes=plt.subplots(3,4,figsize=(16,11))

    # Row 0: maze + input channels
    img=np.stack([maze.astype(float)]*3,axis=-1)
    for pr,pc in path: img[pr,pc]=[0.7,0.7,0.7]
    img[r,c]=[0,0,1]; img[19,20]=[0,1,0]
    axes[0,0].imshow(img); axes[0,0].set_title(f'Maze (step {len(path)})')
    ch_names=['Agent','Exit','Visited']
    for ch in range(3):
        axes[0,ch+1].imshow(obs[0,ch+1],cmap='viridis')
        axes[0,ch+1].set_title(f'Input: {ch_names[ch]}')

    # Row 1-2: feature maps
    layer_names=['Conv1 (32ch)','Conv2 (64ch)','Conv3 (64ch)','Conv4 (128ch)']
    for li,act in enumerate(acts):
        best=act.reshape(act.shape[0],-1).max(axis=1).argmax()
        row = 1 + li//4
        col = li % 4
        axes[1,li].imshow(act[best],cmap='viridis',interpolation='nearest')
        axes[1,li].set_title(f'{layer_names[li]}\nfilter {best}',fontsize=9)

    # Row 2: more conv1 filters + policy
    act0=acts[0]
    activity=act0.reshape(act0.shape[0],-1).max(axis=1)
    top3=np.argsort(activity)[-4:-1][::-1]
    for fi,filt_idx in enumerate(top3):
        axes[2,fi].imshow(act0[filt_idx],cmap='viridis')
        axes[2,fi].set_title(f'Conv1 filter {filt_idx}',fontsize=9)

    action_names=['Down','Right','Up','Left']
    best_a=np.argmax(probs)
    colors=['#e74c3c' if i==best_a else '#3498db' for i in range(4)]
    axes[2,3].barh(action_names,probs,color=colors)
    axes[2,3].set_xlim(0,1)
    axes[2,3].set_title(f'Policy (V={value.item():.1f})',fontsize=9)

    for ax in axes.flat: ax.set_xticks([]);ax.set_yticks([])
    axes[2,3].set_yticks(range(4)); axes[2,3].set_yticklabels(action_names)
    fig.suptitle(f'CNN Intermediate Outputs — pos=({r},{c}), action={action_names[best_a]}',fontsize=12)
    plt.tight_layout(); plt.show()

for _ in range(3):
    idx=np.random.randint(len(eval_mazes))
    visualize_cnn_features_static(model, eval_mazes[idx])

---
## 9.3 Странные лабиринты (до +2)

Модель обучалась только на Kruskal-лабиринтах. Проверяем, обобщается ли она
на лабиринты с нестандартной структурой: случайные препятствия, комнаты, спирали, диагонали.

Используется **та же модель** без дообучения.

In [ ]:
def make_random_obstacles_maze(density=0.3):
    """Открытое пространство со случайными стенами."""
    maze = np.ones((21,21), dtype=np.int32)
    maze[0,:]=0; maze[-1,:]=0; maze[:,0]=0; maze[:,-1]=0
    for r in range(1,20):
        for c in range(1,20):
            if random.random() < density:
                maze[r,c] = 0
    maze[1,0]=1; maze[1,1]=1; maze[19,20]=1; maze[19,19]=1
    sp = bfs_shortest_path(maze)
    return maze if sp.sum() >= 2 else None


def make_rooms_maze():
    """Комнаты соединённые узкими коридорами."""
    maze = np.zeros((21,21), dtype=np.int32)
    rooms = [(1,1,7,7), (1,11,7,7), (11,1,7,7), (11,11,7,7)]
    for rr,cc,h,w in rooms:
        for dr in range(h):
            for dc in range(w):
                if rr+dr<21 and cc+dc<21:
                    maze[rr+dr,cc+dc] = 1
    for r in range(1,20): maze[r,9]=1; maze[r,10]=1
    for c in range(1,20): maze[9,c]=1; maze[10,c]=1
    maze[1,0]=1; maze[19,20]=1
    sp = bfs_shortest_path(maze)
    return maze if sp.sum() >= 2 else None


def make_spiral_maze():
    """Спиральный коридор."""
    maze = np.zeros((21,21), dtype=np.int32)
    r,c = 1,0; maze[r,c]=1
    dirs = [(0,1),(1,0),(0,-1),(-1,0)]
    di=0; seg_len=19; shrink=0
    for _ in range(400):
        if seg_len <= 0: break
        dr,dc = dirs[di%4]
        for _ in range(seg_len):
            nr,nc = r+dr,c+dc
            if 0<=nr<21 and 0<=nc<21:
                maze[nr,nc]=1; r,c=nr,nc
            else: break
        di+=1; shrink+=1
        if shrink%2==0: seg_len-=2
    maze[1,0]=1; maze[19,20]=1
    for _ in range(25):
        maze[random.randint(1,19),random.randint(1,19)]=1
    sp = bfs_shortest_path(maze)
    return maze if sp.sum() >= 2 else None


def make_diagonal_maze():
    """Диагональные коридоры."""
    maze = np.zeros((21,21), dtype=np.int32)
    for i in range(21):
        for d in [-1,0,1]:
            j = i+d
            if 0<=j<21: maze[i,j]=1
            k = 20-i+d
            if 0<=k<21: maze[i,k]=1
    for c in range(21): maze[5,c]=1; maze[10,c]=1; maze[15,c]=1
    for _ in range(30): maze[random.randint(1,19),random.randint(1,19)]=1
    maze[1,0]=1; maze[19,20]=1
    sp = bfs_shortest_path(maze)
    return maze if sp.sum() >= 2 else None


def make_wide_corridors_maze():
    """Широкие коридоры (не типичный лабиринт)."""
    maze = np.zeros((21,21), dtype=np.int32)
    # Горизонтальные полосы шириной 3
    for band_start in [1,5,9,13,17]:
        for r in range(band_start, min(band_start+3,20)):
            for c in range(1,20): maze[r,c]=1
    # Вертикальные соединения
    for c in [3,10,17]:
        for r in range(1,20): maze[r,c]=1
    maze[1,0]=1; maze[19,20]=1
    sp = bfs_shortest_path(maze)
    return maze if sp.sum() >= 2 else None


def generate_strange_mazes(n_per_type=10):
    generators = [
        ('Random sparse', lambda: make_random_obstacles_maze(0.2)),
        ('Random dense',  lambda: make_random_obstacles_maze(0.35)),
        ('Rooms',         make_rooms_maze),
        ('Spiral',        make_spiral_maze),
        ('Diagonal',      make_diagonal_maze),
        ('Wide corridors', make_wide_corridors_maze),
    ]
    mazes, labels = [], []
    for name, gen_fn in generators:
        count, attempts = 0, 0
        while count < n_per_type and attempts < n_per_type*50:
            attempts += 1
            m = gen_fn()
            if m is not None:
                mazes.append(m); labels.append(name); count += 1
        print(f'  {name}: {count}')
    return np.stack(mazes), labels

print('Strange maze generators ready')

In [ ]:
print('Generating strange mazes...')
strange_mazes, strange_labels = generate_strange_mazes(10)
strange_dist, strange_sp = precompute(strange_mazes)

# Показать примеры
types = list(dict.fromkeys(strange_labels))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ti, mtype in enumerate(types[:6]):
    idx = strange_labels.index(mtype)
    ax = axes[ti//3, ti%3]
    ax.imshow(strange_mazes[idx], cmap='binary_r')
    ax.plot(0,1,'go',ms=8); ax.plot(20,19,'ro',ms=8)
    sp_len = int(strange_sp[idx].sum())
    ax.set_title(f'{mtype} (SP={sp_len})', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Странные лабиринты', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Оценка по типам
print('='*50)
print('Оценка модели на странных лабиринтах')
print('(модель обучена ТОЛЬКО на Kruskal-лабиринтах)')
print('='*50)

types = list(dict.fromkeys(strange_labels))
for mtype in types:
    idxs = [i for i,l in enumerate(strange_labels) if l==mtype]
    subset = strange_mazes[idxs]
    subset_d = strange_dist[idxs]
    subset_s = strange_sp[idxs]
    sr, avs = evaluate_model(model, subset, subset_d, subset_s,
                              stage=1, n=len(subset), n_runs=3)
    sp_avg = subset_s.sum(axis=(1,2)).mean()
    print(f'  {mtype:20s}: Success={sr:.0%}, Steps={avs:.0f} (SP={sp_avg:.0f})')

# Общая
sr_all, avs_all = evaluate_model(model, strange_mazes, strange_dist, strange_sp,
                                  stage=1, n=len(strange_mazes), n_runs=3)
print(f'  {"OVERALL":20s}: Success={sr_all:.0%}, Steps={avs_all:.0f}')

In [ ]:
# Визуализация решений на странных лабиринтах
types_shown = set()
for i in range(len(strange_mazes)):
    if strange_labels[i] not in types_shown:
        types_shown.add(strange_labels[i])
        print(f'\n{strange_labels[i]}:')
        visualize_solution(model, strange_mazes[i], strange_sp[i])
    if len(types_shown) >= 6:
        break